# 05b Audit Hydrology

ADR-0016 Wflow Readiness: validate the Wflow discharge **response** against observed USGS instantaneous (IV) records for **historical validation events** (design rows whose rainfall analog runs at ~unit scale — real observed storms), scoring simulated-vs-observed hourly hydrographs (KGE/NSE, peak/volume bias). Also reads the single-K Same-Frequency Amplification provenance and the baseflow validation. Discharge is generated from rainfall + antecedent moisture; there is no injected streamflow member.

## Runtime

Load the current location config, catalog, readiness tables, and Wflow/SFINCS paths.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

try:
    import geopandas as gpd
except Exception as exc:
    gpd = None
    print(f"geopandas unavailable: {exc}")

try:
    import xarray as xr
except Exception as exc:
    xr = None
    print(f"xarray unavailable: {exc}")

notebook_root = Path.cwd().resolve()
if (notebook_root / "config.yaml").exists():
    location_root = notebook_root
elif (notebook_root.parent / "config.yaml").exists():
    location_root = notebook_root.parent
else:
    location_root = Path("locations/greensboro").resolve()

repo_root = location_root.parents[1]
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from study_location import define_location
from wflow_runs.dynamic_handoff import dynamic_handoff_paths, plan_dynamic_wflow_handoff
from wflow_runs.dynamic_handoff_batch import run_dynamic_handoff_batch

definition = define_location(location_root / "config.yaml")
config = definition.config
config["wflow"]["domain_set"]["review_required"] = False
# ADR-0016: USGS IV records are calibration/validation inputs (read directly by this audit),
# not runtime forcing. No fetch/require_instantaneous flags.
config.setdefault("wflow", {}).setdefault("streamflow_calibration", {}).setdefault(
    "event_records_root", "data/sources/usgs_streamgages/event_streamflow_iv"
)
location_name = location_root.name

scenario_catalog_path = location_root / "data/event_catalog/catalog/scenario_catalog.csv"
sfincs_scenarios_root = location_root / "data/sfincs/scenarios"
events_root = location_root / config.get("wflow", {}).get("events_root", "data/wflow/events")
wflow_base_root = location_root / config.get("wflow", {}).get("base_model_root", "data/wflow/base")
readiness_path = sfincs_scenarios_root / f"{location_name}_dynamic_handoff_readiness.csv"
blocked_path = sfincs_scenarios_root / f"{location_name}_blocked_dynamic_handoffs.csv"
accepted_path = sfincs_scenarios_root / f"{location_name}_accepted_dynamic_handoffs.csv"
joint_worklist_path = sfincs_scenarios_root / f"{location_name}_joint_wflow_sfincs_worklist.csv"
streamflow_records_path = location_root / "data/sources/usgs_streamgages/streamflow_records.csv"
event_streamflow_iv_root = location_root / "data/sources/usgs_streamgages/event_streamflow_iv"
audit_plots_dir = location_root / "data/wflow/audit/plots"
audit_plots_dir.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 160)
print(location_root)


## Current Scenario Sample

Randomly sample from the current runnable scenario set. `blocked` means pending Wflow handoff execution; `accepted` means a handoff artifact already exists. `incompatible` rows are excluded from the sample and should normally be zero after catalog repair.


In [ ]:
# ADR-0016: validate Wflow against HISTORICAL events, not a random synthetic sample. Design rows
# whose rainfall analog runs at ~unit scale (rainfall_scale_factor approx 1) ARE real observed
# storms, so Wflow's hydrograph can be scored against observed USGS IV. A config list
# (wflow.validation.historical_event_ids) overrides the heuristic for named storms.
VALIDATION_EVENT_N = 5

catalog = pd.read_csv(scenario_catalog_path, dtype={"event_id": str})
readiness = pd.read_csv(readiness_path, dtype={"event_id": str}) if readiness_path.exists() else pd.DataFrame(columns=["event_id", "status"])
blocked = pd.read_csv(blocked_path, dtype={"event_id": str}) if blocked_path.exists() else pd.DataFrame(columns=["event_id"])
accepted = pd.read_csv(accepted_path, dtype={"event_id": str}) if accepted_path.exists() else pd.DataFrame(columns=["event_id"])
joint_worklist = pd.read_csv(joint_worklist_path, dtype={"event_id": str}) if joint_worklist_path.exists() else pd.DataFrame(columns=["event_id"])

explicit_validation_ids = (config.get("wflow", {}).get("validation", {}) or {}).get("historical_event_ids") or []
if explicit_validation_ids:
    audit_scenarios = catalog[catalog["event_id"].astype(str).isin([str(e) for e in explicit_validation_ids])].copy()
    selection_basis = "config wflow.validation.historical_event_ids"
elif "rainfall_scale_factor" in catalog:
    pool = catalog.copy()
    pool["historical_proximity"] = (pd.to_numeric(pool["rainfall_scale_factor"], errors="coerce") - 1.0).abs()
    audit_scenarios = pool.nsmallest(VALIDATION_EVENT_N, "historical_proximity").sort_values("event_id").reset_index(drop=True)
    selection_basis = "nearest-to-historical (rainfall_scale_factor approx 1.0)"
else:
    audit_scenarios = catalog.sort_values("sample_rp_years", ascending=False).head(VALIDATION_EVENT_N).reset_index(drop=True)
    selection_basis = "largest sample_rp_years (fallback)"
WFLOW_AUDIT_EVENT_IDS = audit_scenarios["event_id"].astype(str).tolist()

summary = pd.Series(
    {
        "catalog_rows": len(catalog),
        "readiness_rows": len(readiness),
        "joint_worklist_rows": len(joint_worklist),
        "validation_event_selection": selection_basis,
        "validation_event_count": len(audit_scenarios),
    },
    name="wflow_validation_inputs",
)
display(summary)
display(audit_scenarios[[column for column in [
    "event_id", "sample_rp_years", "severity_band", "rainfall_member_id", "rainfall_member_time", "rainfall_scale_factor",
] if column in audit_scenarios]])


## Wflow Forcing Completeness

Check that the sampled catalog rows carry the file/time fields Wflow needs to stage event precipitation.


In [ ]:
REQUIRED_WFLOW_FORCING_COLUMNS = ["rainfall_member_file", "rainfall_member_id", "rainfall_member_time"]

missing_forcing_rows = []
for event_id in WFLOW_AUDIT_EVENT_IDS:
    row = catalog.loc[catalog["event_id"].astype(str) == str(event_id)].iloc[0]
    for column in REQUIRED_WFLOW_FORCING_COLUMNS:
        value = row.get(column, pd.NA)
        if pd.isna(value) or str(value).strip() == "":
            missing_forcing_rows.append({"event_id": event_id, "missing_field": column})

forcing_completeness = pd.DataFrame(missing_forcing_rows)
display(
    pd.Series(
        {
            "sampled_events_checked": len(WFLOW_AUDIT_EVENT_IDS),
            "missing_wflow_forcing_values": len(forcing_completeness),
        },
        name="wflow_forcing_completeness",
    )
)
display(forcing_completeness)
if not forcing_completeness.empty:
    raise ValueError("Sampled catalog rows are missing required Wflow forcing fields")


## Rainfall-Driven Readiness (ADR-0016)

Discharge is the Wflow response (no injected streamflow member). Validate the rainfall-driven readiness contract for the selected historical validation events: a rainfall member is wired and the Same-Frequency Amplification / baseflow Primary Reference Gage is configured.

In [ ]:
from wflow_runs.streamflow_realization import validate_wflow_streamflow_realization

readiness_rows = []
for event_id in WFLOW_AUDIT_EVENT_IDS:
    report = validate_wflow_streamflow_realization(
        config, location_root, event_id, catalog_path=scenario_catalog_path, raise_on_error=False
    )
    for _, r in report.iterrows():
        readiness_rows.append({"event_id": event_id, **r.to_dict()})
readiness_diagnosis = pd.DataFrame(readiness_rows)
display(readiness_diagnosis)
if not readiness_diagnosis.empty and readiness_diagnosis["status"].eq("failed").any():
    raise RuntimeError("Validation events are missing required rainfall forcing for Wflow generation.")


## Handoff Hydrograph Diagnostics

Summarize existing event and zero-rain SFINCS handoff hydrographs for the random sample and any already completed local Wflow events.


In [ ]:
domain_manifest_path = location_root / "data/sfincs/domains/domain_set.yaml"
domain_manifest = yaml.safe_load(domain_manifest_path.read_text(encoding="utf-8")) or {}
expected_handoff_ids = []
active_sfincs_domain_id = "greensboro_rural"
for domain in domain_manifest.get("domains", []):
    if str(domain.get("sfincs_domain_id")) == active_sfincs_domain_id:
        expected_handoff_ids.extend([str(value) for value in domain.get("handoff_source_ids", [])])
expected_handoff_ids = list(dict.fromkeys(expected_handoff_ids))

wflow_manifest_path = location_root / "data/wflow/domain_set.yaml"
wflow_manifest = yaml.safe_load(wflow_manifest_path.read_text(encoding="utf-8")) if wflow_manifest_path.exists() else {}
active_wflow_submodel_ids = [
    str(item["wflow_submodel_id"])
    for item in (wflow_manifest or {}).get("submodels", [])
    if item.get("wflow_submodel_id")
]
ACTIVE_WFLOW_SUBMODEL_ID = active_wflow_submodel_ids[0] if active_wflow_submodel_ids else "greensboro_rural"

expected_handoff_ids, ACTIVE_WFLOW_SUBMODEL_ID


In [ ]:
def discharge_frame(discharge_nc):
    if xr is None:
        raise RuntimeError("xarray is required to read discharge NetCDF files")
    with xr.open_dataset(discharge_nc) as ds:
        if "discharge" not in ds:
            raise ValueError(f"{discharge_nc} lacks discharge variable")
        da = ds["discharge"]
        if set(da.dims) >= {"index", "time"}:
            values = da.transpose("time", "index").to_pandas()
        else:
            values = da.to_pandas()
        if "name" in ds and len(ds["name"].values) == values.shape[1]:
            values.columns = [str(value) for value in ds["name"].values]
        else:
            values.columns = [str(col) for col in values.columns]
    values.index = pd.DatetimeIndex(values.index)
    return values.astype(float)

def summarize_handoff_hydrographs(event_id, kind, path):
    path = Path(path)
    if not path.exists():
        return [], []
    frame = discharge_frame(path)
    rows = []
    for source_id, series in frame.items():
        clean = series.dropna()
        rows.append(
            {
                "event_id": event_id,
                "kind": kind,
                "source_id": source_id,
                "peak_m3s": float(clean.max()) if not clean.empty else np.nan,
                "peak_time": clean.idxmax() if not clean.empty else pd.NaT,
                "volume_m3h": float(clean.sum()) if not clean.empty else np.nan,
                "nonzero_steps": int((clean.abs() > 0).sum()) if not clean.empty else 0,
                "path": str(path),
            }
        )
    pair_rows = []
    if frame.shape[1] >= 2:
        corr = frame.corr()
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i + 1:]:
                right_peak = frame[right].max()
                pair_rows.append(
                    {
                        "event_id": event_id,
                        "kind": kind,
                        "left_source_id": left,
                        "right_source_id": right,
                        "shape_correlation": float(corr.loc[left, right]),
                        "peak_ratio_left_over_right": float(frame[left].max() / right_peak) if right_peak else np.nan,
                    }
                )
    return rows, pair_rows

hydrograph_event_ids = list(WFLOW_AUDIT_EVENT_IDS)

hydro_rows = []
pair_rows = []
for event_id in hydrograph_event_ids:
    paths = dynamic_handoff_paths(config, location_root, event_id)
    for kind, path in [("event", paths["discharge"]), ("zero_rain", paths["zero_rain_discharge"] )]:
        rows, pairs = summarize_handoff_hydrographs(event_id, kind, path)
        hydro_rows.extend(rows)
        pair_rows.extend(pairs)

hydrograph_summary = pd.DataFrame(hydro_rows)
hydrograph_pairs = pd.DataFrame(pair_rows)
display(pd.Series({"hydrograph_event_ids_checked": hydrograph_event_ids}, name="handoff_hydrograph_inputs"))
display(hydrograph_summary)
display(hydrograph_pairs)


In [ ]:
if hydrograph_summary.empty:
    zero_rain_comparison = pd.DataFrame()
else:
    pivot = hydrograph_summary.pivot_table(index=["event_id", "source_id"], columns="kind", values="peak_m3s", aggfunc="first")
    if {"event", "zero_rain"}.issubset(set(pivot.columns)):
        pivot["zero_over_event_peak_fraction"] = pivot["zero_rain"] / pivot["event"]
    zero_rain_comparison = pivot.reset_index()
zero_rain_comparison


In [ ]:
def plot_handoff_discharge(event_id):
    paths = dynamic_handoff_paths(config, location_root, event_id)
    frames = {}
    for kind, path in [("event", paths["discharge"]), ("zero_rain", paths["zero_rain_discharge"] )]:
        if Path(path).exists():
            frames[kind] = discharge_frame(path)
    if not frames:
        print(f"{event_id}: no sfincs_discharge.nc files found yet")
        return None

    source_ids = expected_handoff_ids or sorted({col for frame in frames.values() for col in frame.columns})
    fig, axes = plt.subplots(len(source_ids), 1, figsize=(11, max(2.8, 2.4 * len(source_ids))), sharex=True)
    if len(source_ids) == 1:
        axes = [axes]
    for ax, source_id in zip(axes, source_ids):
        for kind, frame in frames.items():
            if source_id in frame:
                ax.plot(frame.index, frame[source_id], label=kind, linewidth=1.8)
        ax.set_title(f"{event_id} | {source_id}")
        ax.set_ylabel("m3/s")
        ax.grid(True, alpha=0.25)
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("time")
    fig.tight_layout()
    fig.savefig(audit_plots_dir / f"{event_id}_handoff_discharge_audit.png", dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return fig



## Restart State And Optional Wflow Audit Run

The sampled Wflow runs use the production dynamic handoff path with the configured warmup/restart state, event meteorology, external streamflow realization, and SFINCS handoff gauges.


In [ ]:
restart_state_path = wflow_base_root / ACTIVE_WFLOW_SUBMODEL_ID / "instate" / "instates.nc"
baseline_root = location_root / config.get("wflow", {}).get("dynamic_handoff", {}).get("baseline_root", "data/wflow/warmup/baseline_90d")
restart_state_readiness = pd.Series(
    {
        "active_wflow_submodel": ACTIVE_WFLOW_SUBMODEL_ID,
        "restart_state_path": str(restart_state_path),
        "restart_state_exists": restart_state_path.exists(),
        "baseline_root": str(baseline_root),
        "baseline_root_exists": baseline_root.exists(),
        "state_policy": config.get("wflow", {}).get("dynamic_handoff", {}).get("state_policy"),
        "warmup_days": config.get("wflow", {}).get("dynamic_handoff", {}).get("warmup_days"),
        "baseline_reference_time": config.get("wflow", {}).get("dynamic_handoff", {}).get("baseline_reference_time"),
    },
    name="restart_state_readiness",
)
restart_state_readiness


In [ ]:
EXECUTE_WFLOW_AUDIT = True
FORCE_WFLOW_AUDIT = True
OVERWRITE_METEO_AUDIT = True

display(pd.Series({"wflow_validation_event_ids": WFLOW_AUDIT_EVENT_IDS}, name="historical_validation_events"))

warmup_audit_plan = pd.DataFrame(
    [plan_dynamic_wflow_handoff(config, location_root, event_id, catalog_path=scenario_catalog_path).to_dict() for event_id in WFLOW_AUDIT_EVENT_IDS]
)
display(warmup_audit_plan)

if EXECUTE_WFLOW_AUDIT:
    audit_reports = []
    for event_id in WFLOW_AUDIT_EVENT_IDS:
        report = run_dynamic_handoff_batch(
            config,
            location_root,
            catalog_path=scenario_catalog_path,
            event_ids=[event_id],
            status="all",
            execute=True,
            force=FORCE_WFLOW_AUDIT,
            overwrite_meteo=OVERWRITE_METEO_AUDIT,
        )
        audit_reports.append(report)
        display(report)
    wflow_audit_report = pd.concat(audit_reports, ignore_index=True) if audit_reports else pd.DataFrame()
else:
    wflow_audit_report = pd.DataFrame(
        {
            "event_id": WFLOW_AUDIT_EVENT_IDS,
            "status": "not_run",
            "message": "Set EXECUTE_WFLOW_AUDIT=True to run the production dynamic Wflow handoff for the historical validation events.",
        }
    )
wflow_audit_report


## Post-Run Handoff Diagnostics

After Wflow execution, inspect event and zero-rain handoff hydrographs for the random sample.


In [ ]:
for event_id in WFLOW_AUDIT_EVENT_IDS:
    plot_handoff_discharge(event_id)


In [ ]:
def total_discharge_peak(discharge_nc):
    if xr is None:
        raise RuntimeError("xarray is required to read discharge NetCDF files")
    with xr.open_dataset(discharge_nc) as ds:
        da = ds["discharge"]
        if "index" in da.dims:
            da = da.sum("index")
        return float(da.max(skipna=True))

post_run_hydro_rows = []
for event_id in WFLOW_AUDIT_EVENT_IDS:
    paths = dynamic_handoff_paths(config, location_root, event_id)
    event_rows, _ = summarize_handoff_hydrographs(event_id, "event", paths["discharge"])
    zero_rows, _ = summarize_handoff_hydrographs(event_id, "zero_rain", paths["zero_rain_discharge"])
    event_peak = total_discharge_peak(paths["discharge"]) if paths["discharge"].exists() else np.nan
    zero_peak = total_discharge_peak(paths["zero_rain_discharge"]) if paths["zero_rain_discharge"].exists() else np.nan
    post_run_hydro_rows.append(
        {
            "event_id": event_id,
            "event_source_count": len(event_rows),
            "zero_rain_source_count": len(zero_rows),
            "event_total_peak_m3s": event_peak,
            "zero_rain_total_peak_m3s": zero_peak,
            "zero_over_event_peak_fraction": zero_peak / event_peak if event_peak and np.isfinite(event_peak) else np.nan,
            "event_discharge_exists": paths["discharge"].exists(),
            "zero_rain_discharge_exists": paths["zero_rain_discharge"].exists(),
        }
    )
post_run_zero_rain_diagnostics = pd.DataFrame(post_run_hydro_rows)
display(post_run_zero_rain_diagnostics)

# ADR-0016 readout: single-K Same-Frequency Amplification provenance + baseflow Wflow Readiness.
from wflow_runs.coupling_qa import validate_baseflow_against_observed

amp_rows = []
baseflow_frames = []
for event_id in WFLOW_AUDIT_EVENT_IDS:
    paths = dynamic_handoff_paths(config, location_root, event_id)
    amp_json = Path(paths["discharge"]).with_name("sfincs_discharge.amplification.json")
    if amp_json.exists():
        amp = json.loads(amp_json.read_text(encoding="utf-8"))
        amp_rows.append({"event_id": event_id, "K": amp.get("K"), "K_status": amp.get("status"), "reference_gage": amp.get("reference_gage")})
    zero_rain_nc = paths["zero_rain_discharge"]
    if Path(zero_rain_nc).exists():
        bf = validate_baseflow_against_observed(
            config, location_root, zero_rain_discharge_nc=zero_rain_nc, streamflow_records_csv=streamflow_records_path
        )
        bf.insert(0, "event_id", event_id)
        baseflow_frames.append(bf)

amplification_readout = pd.DataFrame(amp_rows)
baseflow_readout = pd.concat(baseflow_frames, ignore_index=True) if baseflow_frames else pd.DataFrame()
display(pd.Series({"single_K_amplification": "per-event provenance (ADR-0016)"}, name="amplification"))
display(amplification_readout)
display(pd.Series({"baseflow_validation": "zero-rain control vs observed low-flow at Primary Reference Gage"}, name="baseflow"))
display(baseflow_readout)


## Wflow Gauge Output And USGS Comparison

Read Wflow native scalar outputs, map `Q_<index>` columns back to SFINCS and USGS gauge layers, and compare simulated/observed hydrographs where records overlap.


In [ ]:
CFS_TO_CMS = 0.028316846592


def read_wflow_output_csv(event_id, submodel_id=None, *, control=False):
    submodel_id = submodel_id or ACTIVE_WFLOW_SUBMODEL_ID
    event_root = events_root / str(event_id)
    model_root = event_root / ("_zero_rain" if control else "") / submodel_id if control else event_root / submodel_id
    csv_path = model_root / "run_event" / "output.csv"
    if not csv_path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(csv_path, parse_dates=["time"]).set_index("time")
    frame.index = pd.DatetimeIndex(frame.index)
    return frame


def gauge_output_map(event_id, layer="gauges_usgs", submodel_id=None):
    submodel_id = submodel_id or ACTIVE_WFLOW_SUBMODEL_ID
    gauges_path = events_root / str(event_id) / submodel_id / "staticgeoms" / f"{layer}.geojson"
    if not gauges_path.exists():
        gauges_path = location_root / "data/wflow/base" / submodel_id / "staticgeoms" / f"{layer}.geojson"
    if not gauges_path.exists() or gpd is None:
        return pd.DataFrame()
    gauges = gpd.read_file(gauges_path)
    gauges["q_column"] = "Q_" + gauges["index"].astype(int).astype(str)
    return pd.DataFrame(gauges.drop(columns="geometry"))


def event_iv_records(event_id):
    files = sorted(event_streamflow_iv_root.glob(f"{event_id}_*.csv"))
    if not files:
        return pd.DataFrame(columns=["site_no", "time", "discharge_cfs", "source"])
    frames = [pd.read_csv(path, dtype={"site_no": str}, parse_dates=["time"]) for path in files]
    records = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["site_no", "time"])
    records["source"] = records.get("source", pd.Series("usgs_iv", index=records.index)).fillna("usgs_iv")
    return records.sort_values(["site_no", "time"])


def observed_event_site_flow(event_id, site_no, start, end):
    records = event_iv_records(event_id)
    selected = records[records["site_no"].astype(str).eq(str(site_no))].copy()
    if selected.empty:
        return pd.Series(dtype=float, name=str(site_no))
    selected = selected.set_index("time").sort_index()
    observed = selected["discharge_cfs"].astype(float) * CFS_TO_CMS
    observed = observed.loc[(observed.index >= pd.Timestamp(start)) & (observed.index <= pd.Timestamp(end))]
    return observed.resample("1h").mean().interpolate("time", limit_area="inside").rename(str(site_no))


def hydrograph_scores(simulated, observed):
    joined = pd.concat([simulated.rename("sim"), observed.rename("obs")], axis=1).dropna()
    if joined.empty:
        return {"n": 0, "nse": np.nan, "kge": np.nan, "peak_bias_fraction": np.nan, "volume_bias_fraction": np.nan}
    obs = joined["obs"].to_numpy(dtype=float)
    sim = joined["sim"].to_numpy(dtype=float)
    denom = np.sum((obs - obs.mean()) ** 2)
    nse = 1.0 - np.sum((sim - obs) ** 2) / denom if denom else np.nan
    r = np.corrcoef(sim, obs)[0, 1] if len(joined) > 1 and np.std(sim) and np.std(obs) else np.nan
    alpha = np.std(sim) / np.std(obs) if np.std(obs) else np.nan
    beta = np.mean(sim) / np.mean(obs) if np.mean(obs) else np.nan
    kge = 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2) if np.isfinite([r, alpha, beta]).all() else np.nan
    return {
        "n": int(len(joined)),
        "nse": float(nse) if np.isfinite(nse) else np.nan,
        "kge": float(kge) if np.isfinite(kge) else np.nan,
        "peak_bias_fraction": float((sim.max() - obs.max()) / obs.max()) if obs.max() else np.nan,
        "volume_bias_fraction": float((sim.sum() - obs.sum()) / obs.sum()) if obs.sum() else np.nan,
    }


def usgs_calibration_table(event_id):
    sim = read_wflow_output_csv(event_id)
    gauges = gauge_output_map(event_id, layer="gauges_usgs")
    iv = event_iv_records(event_id)
    if sim.empty or gauges.empty or iv.empty:
        return pd.DataFrame()
    rows = []
    for _, gauge in gauges.iterrows():
        q_col = str(gauge["q_column"])
        if q_col not in sim:
            continue
        obs = observed_event_site_flow(event_id, gauge["site_no"], sim.index.min(), sim.index.max())
        scores = hydrograph_scores(sim[q_col].astype(float), obs)
        rows.append({"event_id": event_id, "site_no": str(gauge["site_no"]), "q_column": q_col, **scores})
    return pd.DataFrame(rows)

calibration_event_ids = list(WFLOW_AUDIT_EVENT_IDS)
calibration_tables = [usgs_calibration_table(event_id) for event_id in calibration_event_ids]
calibration_summary = pd.concat([table for table in calibration_tables if not table.empty], ignore_index=True) if any(not table.empty for table in calibration_tables) else pd.DataFrame()
calibration_summary


In [ ]:
def plot_wflow_sfincs_gauge_output(event_id):
    sim = read_wflow_output_csv(event_id)
    gauges = gauge_output_map(event_id, layer="gauges_sfincs")
    if sim.empty or gauges.empty:
        print(f"{event_id}: missing Wflow run_event/output.csv or gauges_sfincs.geojson")
        return None
    fig, ax = plt.subplots(figsize=(11, 4.5))
    plotted = False
    for _, gauge in gauges.sort_values("sfincs_handoff_id").iterrows():
        q_col = str(gauge["q_column"])
        label = str(gauge.get("sfincs_handoff_id") or gauge.get("name") or q_col)
        if q_col in sim:
            ax.plot(sim.index, sim[q_col].astype(float), label=label, linewidth=1.8)
            plotted = True
    if not plotted:
        print(f"{event_id}: no gauges_sfincs Q columns found in output.csv")
        plt.close(fig)
        return None
    ax.set_title(f"{event_id} | Wflow gauges_sfincs discharge")
    ax.set_ylabel("m3/s")
    ax.set_xlabel("time")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(audit_plots_dir / f"{event_id}_wflow_sfincs_gauge_output.png", dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return fig

for event_id in WFLOW_AUDIT_EVENT_IDS:
    plot_wflow_sfincs_gauge_output(event_id)


In [ ]:
def plot_usgs_observation_comparison(event_id, *, max_gauges=6):
    sim = read_wflow_output_csv(event_id)
    gauges = gauge_output_map(event_id, layer="gauges_usgs")
    iv = event_iv_records(event_id)
    if sim.empty or gauges.empty:
        print(f"{event_id}: missing Wflow output or gauges_usgs layer")
        return None
    if iv.empty:
        print(f"{event_id}: no cached USGS IV event-window records under {event_streamflow_iv_root}")
        return None
    rows = []
    for _, gauge in gauges.iterrows():
        q_col = str(gauge["q_column"])
        if q_col not in sim:
            continue
        obs = observed_event_site_flow(event_id, gauge["site_no"], sim.index.min(), sim.index.max())
        joined = pd.concat([sim[q_col].astype(float).rename("sim"), obs.rename("obs")], axis=1).dropna()
        if joined.empty:
            continue
        score = hydrograph_scores(joined["sim"], joined["obs"])
        rows.append((str(gauge["site_no"]), q_col, joined, score))
    if not rows:
        print(f"{event_id}: no overlapping USGS IV observed records for plotted gauge outputs")
        return None
    rows = rows[:max_gauges]
    fig, axes = plt.subplots(len(rows), 1, figsize=(11, max(3.0, 2.6 * len(rows))), sharex=True)
    if len(rows) == 1:
        axes = [axes]
    for ax, (site_no, q_col, joined, score) in zip(axes, rows):
        ax.plot(joined.index, joined["sim"], label="Wflow simulated", linewidth=1.8)
        ax.plot(joined.index, joined["obs"], label="USGS IV observed, hourly", linewidth=1.5, alpha=0.85)
        ax.set_title(f"{event_id} | USGS {site_no} | KGE={score['kge']:.2f} NSE={score['nse']:.2f}")
        ax.set_ylabel("m3/s")
        ax.grid(True, alpha=0.25)
        ax.legend(loc="best")
    axes[-1].set_xlabel("time")
    fig.tight_layout()
    fig.savefig(audit_plots_dir / f"{event_id}_usgs_iv_observation_comparison.png", dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return fig

for event_id in WFLOW_AUDIT_EVENT_IDS:
    plot_usgs_observation_comparison(event_id)
